In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time
import json
import re
import string

In [17]:
def to_json(data):
    """
    This function takes list of dictionary as Input and 
    then Creates a JSON file in which Input data is stored
    """
    with open("data_dict.json", "w") as outfile:
        json.dump(data, outfile,indent=4)
        outfile.close()

In [18]:
data_list = []
url = "http://sheriff2citizen.fauquiercounty.gov/wantedlist.aspx#"
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") 
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--log-level=3")

In [19]:
def get_data(slug_name):
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    search = driver.find_element(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[5]/div/table/tbody/tr/td[2]/table/tbody/tr/td[5]/select')
    search.send_keys("All")
    search.send_keys(Keys.RETURN)
    time.sleep(3)
    list1 = driver.find_elements(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[3]/div[3]/div/table/tbody/tr')
    data_dict = {
                    "fullName" : "",
                    "firstName" : "",
                    "lastName" : "",
                    "race" : "",
                    "gender" : "",
                    "age" : "",
                    "offence" : [],
                    "fullAddress" : "",
                    "additionalInfo" : "",
                    "charges" : [],
                    "summary" : ""
            }
    previousName = ""
    for i in range (1, len(list1)+1):
        list2 = driver.find_elements(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[3]/div[3]/div/table/tbody/tr[{i}]/td')
        for j in range (1, len(list2)+1):
            if j==1:
                name = driver.find_element(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[3]/div[3]/div/table/tbody/tr[{i}]/td[{j}]').text
                info = name.split("(")[1]
                fullName = name.split("(")[0]
                firstName = fullName.split(",")[1].strip()
                lastName = fullName.split(",")[0]
                fullName = firstName + " " + lastName
                if i==len(list1)+1:
                    fullName= "Ishaan"
                race = info.split("/")[0]
                gender = info.split("/")[1]
                age = info.split("/")[2].replace(")", "")
            elif j==8:
                charge = driver.find_element(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[3]/div[3]/div/table/tbody/tr[{i}]/td[{j}]').text
            elif j==9:
                offence = driver.find_element(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[3]/div[3]/div/table/tbody/tr[{i}]/td[{j}]').text
            elif j==10:
                address = driver.find_element(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[3]/div[3]/div/table/tbody/tr[{i}]/td[{j}]').text
            elif j==11:
                agency = driver.find_element(By.XPATH, f'/html/body/form/table/tbody/tr[2]/td/div[2]/div[2]/div[2]/div[3]/div[3]/div/table/tbody/tr[{i}]/td[{j}]').text
        additionalInfo = "Arresting agency: " + agency
        if fullName != previousName and len(data_dict['fullName'])>0:
            print(fullName)
            print(previousName)
            print(data_dict['fullName'])
            data_dict['charges'] = list(set(data_dict['charges']))
            data_dict['offence'] = list(set(data_dict['offence']))
            data_dict['offence'] = ", ".join(data_dict['offence'])
            data_dict['charges'] = ", ".join(data_dict['charges'])
            summary = data_dict['fullName'] + " is charged with " + ", "+ (data_dict['charges']) + " for the " + ", "+(data_dict['offence'])
            data_dict['summary'] = summary
            print(summary)
            data_list.append(data_dict)
            data_dict = {
                    "fullName" : "",
                    "firstName" : "",
                    "lastName" : "",
                    "race" : "",
                    "gender" : "",
                    "age" : "",
                    "offence" : [],
                    "fullAddress" : "",
                    "additionalInfo" : "",
                    "charges" : [],
                    "summary": ""
            }
        if fullName and fullName == previousName:
            if offence:
                data_dict['offence'].append(offence)
            if charge:
                data_dict['charges'].append(charge)
        else:
            if fullName:
                data_dict['fullName'] = fullName
            if firstName:
                data_dict['firstName'] = firstName
            if lastName:
                data_dict['lastName'] = lastName
            if offence:
                data_dict['offence'].append(offence)
            if charge:
                data_dict['charges'].append(charge)
            if age:
                data_dict['age'] = age
            if gender:
                data_dict['gender'] = gender
            if race:
                data_dict['race'] = race
            if address:
                data_dict['fullAddress'] = address
            if additionalInfo:
                data_dict['additionalInfo'] = additionalInfo
        if len(list1) == i:
            data_dict['charges'] = list(set(data_dict['charges']))
            data_dict['offence'] = list(set(data_dict['offence']))
            data_dict['offence'] = ", ".join(data_dict['offence'])
            data_dict['charges'] = ", ".join(data_dict['charges'])
            summary = data_dict['fullName'] + " is charged with " + ", "+ (data_dict['charges']) + " for the " + ", "+(data_dict['offence'])
            data_dict['summary'] = summary
            data_list.append(data_dict)
        previousName = fullName
#         data_list.append(data_dict)
#     final_data_list = [i for n, i in enumerate(data_list) if i not in data_list[n + 1:]]
    return data_list

In [20]:
if __name__ == '__main__':
    data_list = get_data("add_slug_name")
    to_json(data_list)

TIMOTHY DANIEL ALEXANDER
JAMES LARRY ALEXANDER
JAMES LARRY ALEXANDER
JAMES LARRY ALEXANDER is charged with , CONTEMPT OF COURT for the , CAPIAS TO SHOW CAUSE
MICHAEL ALLEN
TIMOTHY DANIEL ALEXANDER
TIMOTHY DANIEL ALEXANDER
TIMOTHY DANIEL ALEXANDER is charged with , REVOCATION OF SUSPEND SENTENCE AND PROBA for the , CAPIAS TO SHOW CAUSE
MELVIN ALEXANDER ANDINO RIVERA
MICHAEL ALLEN
MICHAEL ALLEN
MICHAEL ALLEN is charged with , PETIT LARCENY - POCKET PICKING, OBTAINING MONEY BY FALSE PRETENSES, FORGING COIN AND BANK NOTES for the , FELONY, MISDEMEANOR
SANTO ADAM AYALA
MELVIN ALEXANDER ANDINO RIVERA
MELVIN ALEXANDER ANDINO RIVERA
MELVIN ALEXANDER ANDINO RIVERA is charged with , VIOLATE CONDITION OF RELEASE for the , CAPIAS: ATTACHMENT OF THE BODY/CRIMINAL
DEVIN ORVILLE BANKS
SANTO ADAM AYALA
SANTO ADAM AYALA
SANTO ADAM AYALA is charged with , POSSESSION OF CONTROLLED SUBSTANCES for the , FELONY
SHYKE MARQUIS BARKLEY
DEVIN ORVILLE BANKS
DEVIN ORVILLE BANKS
DEVIN ORVILLE BANKS is charged with

JORDAN STORM FILLINGAME
JORGE FIGUEROA RODRIGUEZ
JORGE FIGUEROA RODRIGUEZ
JORGE FIGUEROA RODRIGUEZ is charged with , POSSESSION OF CONTROLLED SUBSTANCES for the , CAPIAS/INDICTMENT (S)
SHELBY RENAE FOLEY
JORDAN STORM FILLINGAME
JORDAN STORM FILLINGAME
JORDAN STORM FILLINGAME is charged with , POSSESSION OF CONTROLLED SUBSTANCES for the , CAPIAS/INDICTMENT (S)
BRIAR KENDRA ADELA FORD
SHELBY RENAE FOLEY
SHELBY RENAE FOLEY
SHELBY RENAE FOLEY is charged with , VIOLATE CONDITION OF RELEASE for the , CAPIAS: ATTACHMENT OF THE BODY/CRIMINAL
JOSHUA ISAAC FOWLER
BRIAR KENDRA ADELA FORD
BRIAR KENDRA ADELA FORD
BRIAR KENDRA ADELA FORD is charged with , CONTEMPT OF COURT for the , CAPIAS: ATTACHMENT OF THE BODY/CRIMINAL
JULIO C FUNES
JOSHUA ISAAC FOWLER
JOSHUA ISAAC FOWLER
JOSHUA ISAAC FOWLER is charged with , REVOCATION OF SUSPEND SENTENCE AND PROBA for the , CAPIAS TO SHOW CAUSE
KENNETH SCOTT FURR
JULIO C FUNES
JULIO C FUNES
JULIO C FUNES is charged with , EMPLOYER FALSE STMNTS WORK COMP for the

TISHANNA MCCOLGAN
MARK ALLEN MCCLUNG
MARK ALLEN MCCLUNG
MARK ALLEN MCCLUNG is charged with , CONTEMPT OF COURT for the , CAPIAS: ATTACHMENT OF THE BODY/CRIMINAL
AUSTIN JOSEPH MCCORKLE
TISHANNA MCCOLGAN
TISHANNA MCCOLGAN
TISHANNA MCCOLGAN is charged with , CONTEMPT OF COURT for the , CAPIAS TO SHOW CAUSE
RYAN PATRICK MCELHINNEY
AUSTIN JOSEPH MCCORKLE
AUSTIN JOSEPH MCCORKLE
AUSTIN JOSEPH MCCORKLE is charged with , CONTEMPT OF COURT for the , CAPIAS TO SHOW CAUSE
ADAM MICHAEL MCGOWAN
RYAN PATRICK MCELHINNEY
RYAN PATRICK MCELHINNEY
RYAN PATRICK MCELHINNEY is charged with , 18.2-272,M for the , MAGISTRATE SUMMONS
KIMBERLY MICHELLE MCKENZIE
ADAM MICHAEL MCGOWAN
ADAM MICHAEL MCGOWAN
ADAM MICHAEL MCGOWAN is charged with , PROHIBITED ACTS BY CONTRACTOR, FAILURE TO DELIVER CROP AFTER ADVANCE, CONSPIRE/AID/ABET: TO COMMIT LARCENY > $200 for the , FELONY, MISDEMEANOR
RACHEL DOREEN MEADOWS
KIMBERLY MICHELLE MCKENZIE
KIMBERLY MICHELLE MCKENZIE
KIMBERLY MICHELLE MCKENZIE is charged with , REVOCATION 

AMANDA LYNN ROS CORBIN
JOSE DANIEL RODRIGUEZ LOPEZ
JOSE DANIEL RODRIGUEZ LOPEZ
JOSE DANIEL RODRIGUEZ LOPEZ is charged with , POSSESSION OF CONTROLLED SUBSTANCES for the , FELONY
JASON VAUGHN ROSSER
AMANDA LYNN ROS CORBIN
AMANDA LYNN ROS CORBIN
AMANDA LYNN ROS CORBIN is charged with , MANUF, SALE, POSSESSION CONTROLLED SUBSTANCE, CONTEMPT OF COURT for the , CAPIAS: ATTACHMENT OF THE BODY/CRIMINAL, CAPIAS
DON JUAN RUCKER
JASON VAUGHN ROSSER
JASON VAUGHN ROSSER
JASON VAUGHN ROSSER is charged with , REVOCATION OF PRETRIAL RELEASE-FELONY for the , CAPIAS TO SHOW CAUSE
ISRAEL VANNESS SHELTON
DON JUAN RUCKER
DON JUAN RUCKER
DON JUAN RUCKER is charged with , OBTAINING MONEY BY FALSE PRETENSES, CREDIT CARD THEFT for the , FELONY, MISDEMEANOR
RAZA HASAN SIKANDAR
ISRAEL VANNESS SHELTON
ISRAEL VANNESS SHELTON
ISRAEL VANNESS SHELTON is charged with , VIOLATE CONDITION OF RELEASE for the , CAPIAS: ATTACHMENT OF THE BODY/CRIMINAL
ANDREW ROBERT SKUDZIENSKI
RAZA HASAN SIKANDAR
RAZA HASAN SIKANDAR
RAZA 